# Colab Full Reproduction

Notebook này dùng cho Colab/Colab extension. Thứ tự chạy được khóa là `Base -> Word2Vec -> CodeBERT`, sau đó mới đến `Enhanced`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!python -m pip install -q --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s eta 0:00:00a 0:00:01


In [3]:
!python -m pip install -q --upgrade pip
!python -m pip install -q pandas numpy scipy scikit-learn gensim sentencepiece accelerate

In [4]:
!python -m pip install -q torch transformers tensorflow

In [ ]:
%cd /content
!rm -rf /content/VTI-Project2
!git clone --depth 1 --branch main https://github.com/tudzct/VTI-Project2.git /content/VTI-Project2
%cd /content/VTI-Project2
!git rev-parse --short HEAD

: 

In [5]:
import os

BASE_OUT = '/content/drive/MyDrive/VTI-Project2-runs'
os.makedirs(BASE_OUT, exist_ok=True)
print(BASE_OUT)

/content/drive/MyDrive/VTI-Project2-runs


In [6]:
!ls -lah artifacts/data_full
!ls -lah artifacts/data_sample

ls: cannot access 'artifacts/data_full': No such file or directory
ls: cannot access 'artifacts/data_sample': No such file or directory


## Base full

Chạy bản thực dụng trước với `p_value_grid = 0.5`.

In [ ]:
!python scripts/run_base_experiment.py \
  --train artifacts/data_full/train.csv.gz \
  --val artifacts/data_full/val.csv.gz \
  --test artifacts/data_full/test.csv.gz \
  --output-dir "$BASE_OUT/base_full_p05" \
  --p-value-grid 0.5

In [ ]:
!ls -lah "$BASE_OUT/base_full_p05"
!cat "$BASE_OUT/base_full_p05/metrics.json"

## Word2Vec full

In [ ]:
!python scripts/run_word2vec_experiment.py \
  --train artifacts/data_full/train.csv.gz \
  --val artifacts/data_full/val.csv.gz \
  --test artifacts/data_full/test.csv.gz \
  --output-dir "$BASE_OUT/word2vec_full" \
  --max-train-rows 0 \
  --max-val-rows 0 \
  --max-test-rows 0 \
  --vector-size 300 \
  --w2v-epochs 30 \
  --nn-epochs 10 \
  --max-len 512 \
  --batch-size 128

In [ ]:
!ls -lah "$BASE_OUT/word2vec_full"
!cat "$BASE_OUT/word2vec_full/metrics.json"

Nếu Word2Vec bị OOM, dùng cell dự phòng bên dưới.

In [ ]:
# !python scripts/run_word2vec_experiment.py \
#   --train artifacts/data_full/train.csv.gz \
#   --val artifacts/data_full/val.csv.gz \
#   --test artifacts/data_full/test.csv.gz \
#   --output-dir "$BASE_OUT/word2vec_full" \
#   --max-train-rows 0 \
#   --max-val-rows 0 \
#   --max-test-rows 0 \
#   --vector-size 200 \
#   --w2v-epochs 20 \
#   --nn-epochs 6 \
#   --max-len 256 \
#   --batch-size 64

## CodeBERT full

In [ ]:
!nvidia-smi

In [ ]:
!python scripts/run_codebert_experiment.py \
  --train artifacts/data_full/train.csv.gz \
  --val artifacts/data_full/val.csv.gz \
  --test artifacts/data_full/test.csv.gz \
  --output-dir "$BASE_OUT/codebert_full" \
  --max-train-rows 0 \
  --max-val-rows 0 \
  --max-test-rows 0 \
  --epochs 2 \
  --batch-size 8 \
  --max-length 256 \
  --learning-rate 2e-5

In [ ]:
!ls -lah "$BASE_OUT/codebert_full"
!cat "$BASE_OUT/codebert_full/metrics.json"

Nếu CodeBERT bị OOM, dùng cell dự phòng bên dưới.

In [ ]:
# !python scripts/run_codebert_experiment.py \
#   --train artifacts/data_full/train.csv.gz \
#   --val artifacts/data_full/val.csv.gz \
#   --test artifacts/data_full/test.csv.gz \
#   --output-dir "$BASE_OUT/codebert_full" \
#   --max-train-rows 0 \
#   --max-val-rows 0 \
#   --max-test-rows 0 \
#   --epochs 1 \
#   --batch-size 4 \
#   --max-length 256 \
#   --learning-rate 2e-5

## Enhanced (chạy sau)

Chỉ bắt đầu khi 3 mô hình nền đã có `raw_preds.csv` và `labels.csv`. Cần upload trước 2 file CPG CSV lên Drive.

In [ ]:
# !python scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds "$BASE_OUT/base_full_p05/raw_preds.csv" \
#   --labels "$BASE_OUT/base_full_p05/labels.csv" \
#   --output-dir "$BASE_OUT/enhanced_base_full"

In [ ]:
# !python scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds "$BASE_OUT/word2vec_full/raw_preds.csv" \
#   --labels "$BASE_OUT/word2vec_full/labels.csv" \
#   --output-dir "$BASE_OUT/enhanced_word2vec_full"

In [ ]:
# !python scripts/run_enhanced_notebook_compat.py \
#   --train "/content/drive/MyDrive/VTI-Project2-joern/train_subset_16384_with_cpg_fullrows.csv" \
#   --test "/content/drive/MyDrive/VTI-Project2-joern/test_full_with_cpg_fullrows.csv" \
#   --raw-preds "$BASE_OUT/codebert_full/raw_preds.csv" \
#   --labels "$BASE_OUT/codebert_full/labels.csv" \
#   --output-dir "$BASE_OUT/enhanced_codebert_full"

In [ ]:
!find "$BASE_OUT" -maxdepth 2 -type f | sort